In [10]:
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup as Soup
import os
import time
import random
import tiktoken
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import TypedDict
# from langchain_core.messages import HumanMessage, AIMessage
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.prompts import ChatPromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor, LLMChainFilter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_experimental.tools import PythonAstREPLTool
from langchain.globals import set_verbose
import uuid
from langchain_core.documents import Document
import httpx
from langgraph.store.postgres import PostgresStore
from psycopg import Connection
import psycopg2
import logging
from pydantic import BaseModel, Field
from typing import Literal, Any, Optional, List, Dict
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from datetime import datetime
from langchain.document_loaders import PyPDFLoader, DirectoryLoader

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv

load_dotenv()

# set_debug(True)
set_verbose(True)

httpx_logger = logging.getLogger("httpx")
httpx_logger.setLevel(logging.WARNING)

llm = AzureChatOpenAI(model_name="gpt-4o", temperature=0, deployment_name = 'gpt-4o', http_client=httpx.Client(verify=False))

openai_embed_model = AzureOpenAIEmbeddings(azure_deployment="text-embedding-3-large",openai_api_version="2024-02-01", http_client=httpx.Client(verify=False))

In [11]:
def load_documents(self, pdf_directory: str) -> List[Document]:
        """Load PDF documents from directory"""
        try:
            loader = DirectoryLoader(
                pdf_directory,
                glob="**/*.pdf",
                loader_cls=PyPDFLoader,
                show_progress=True
            )
            documents = loader.load()
            
            # Clean and preprocess documents
            for doc in documents:
                # Clean text
                doc.page_content = re.sub(r'\s+', ' ', doc.page_content)
                doc.page_content = doc.page_content.strip()
                
                # Add metadata
                if 'source' in doc.metadata:
                    doc.metadata['filename'] = os.path.basename(doc.metadata['source'])
                    doc.metadata['loaded_at'] = datetime.now().isoformat()
            
            self.documents = documents
            logger.info(f"Loaded {len(documents)} documents")
            return documents
            
        except Exception as e:
            logger.error(f"Error loading documents: {str(e)}")
            raise

In [12]:
def split_documents(self, documents: List[Document], chunk_size: int = 1000, 
                       chunk_overlap: int = 200) -> List[Document]:
        """Split documents into chunks"""
        try:
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                length_function=len,
                separators=["\n\n", "\n", " ", ""]
            )
            
            chunks = text_splitter.split_documents(documents)
            
            # Add chunk metadata
            for i, chunk in enumerate(chunks):
                chunk.metadata['chunk_id'] = i
                chunk.metadata['chunk_size'] = len(chunk.page_content)
            
            logger.info(f"Split into {len(chunks)} chunks")
            return chunks
            
        except Exception as e:
            logger.error(f"Error splitting documents: {str(e)}")
            raise

In [13]:
from langchain.embeddings import HuggingFaceEmbeddings

In [14]:
def create_embeddings(self, model_name: str) -> Any:
        """Create embedding model"""
        try:
            if model_name == "openai":
                embeddings = openai_embed_model
            else:
                embeddings = HuggingFaceEmbeddings(
                    model_name=model_name,
                    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
                    encode_kwargs={'normalize_embeddings': True}
                )
            
            self.embedding_model = embeddings
            logger.info(f"Created embedding model: {model_name}")
            return embeddings
            
        except Exception as e:
            logger.error(f"Error creating embeddings: {str(e)}")
            raise

In [15]:
def create_vector_store(self, chunks: List[Document], embeddings: Any) -> FAISS:
    """Create FAISS vector store"""
    try:
        vector_store = FAISS.from_documents(chunks, embeddings)
        self.vector_store = vector_store
        logger.info("Created FAISS vector store")
        return vector_store
        
    except Exception as e:
        logger.error(f"Error creating vector store: {str(e)}")
        raise
    
def save_vector_store(self, vector_store: FAISS, path: str):
    """Save vector store to disk"""
    try:
        vector_store.save_local(path)
        logger.info(f"Saved vector store to {path}")
    except Exception as e:
        logger.error(f"Error saving vector store: {str(e)}")
        raise

def load_vector_store(self, path: str, embeddings: Any) -> FAISS:
    """Load vector store from disk"""
    try:
        vector_store = FAISS.load_local(path, embeddings, allow_dangerous_deserialization=True)
        self.vector_store = vector_store
        logger.info(f"Loaded vector store from {path}")
        return vector_store
    except Exception as e:
        logger.error(f"Error loading vector store: {str(e)}")
        raise

In [21]:
def create_retriever(self, k: int = 5) -> Any:
    """Create retriever based on strategy"""
    try:
        retriever = self.vector_store.as_retriever(search_kwargs={"k": k})
        compressor = LLMChainFilter.from_llm(llm)
        final_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=retriever)
        self.retriever = final_retriever
        logger.info(f"Created retriever ")
        return retriever
    except Exception as e:
        logger.error(f"Error creating retriever: {str(e)}")
        raise

In [17]:
from langchain.chains import RetrievalQA

In [18]:
def create_qa_chain(self, llm: Any, retriever: Any) -> RetrievalQA:
        """Create QA chain"""
        try:
            # Custom prompt template
            template = """Use the following pieces of context to answer the question at the end. 
            If you don't know the answer, just say that you don't know, don't try to make up an answer.
            Always cite the source documents used in your answer.

            Context: {context}

            Question: {question}

            Answer: """
            
            prompt = ChatPromptTemplate(
                template=template,
                input_variables=["context", "question"]
            )
            
            qa_chain = RetrievalQA.from_chain_type(
                llm=llm,
                chain_type="stuff",
                retriever=retriever,
                return_source_documents=True,
                chain_type_kwargs={"prompt": prompt},
                memory=self.memory
            )
            
            self.qa_chain = qa_chain
            logger.info("Created QA chain")
            return qa_chain
            
        except Exception as e:
            logger.error(f"Error creating QA chain: {str(e)}")
            raise

In [19]:
def answer_question(self, question: str) -> Dict[str, Any]:
    """Answer question and return sources"""
    try:
        if not self.qa_chain:
            raise ValueError("QA chain not created")
        
        result = self.qa_chain({"query": question})
        
        # Extract source information
        sources = []
        if 'source_documents' in result:
            for i, doc in enumerate(result['source_documents'][:3]):  # Top 3 sources
                source_info = {
                    'rank': i + 1,
                    'filename': doc.metadata.get('filename', 'Unknown'),
                    'page': doc.metadata.get('page', 'Unknown'),
                    'chunk_id': doc.metadata.get('chunk_id', 'Unknown'),
                    'content_preview': doc.page_content[:200] + "..."
                }
                sources.append(source_info)
        
        return {
            'answer': result['result'],
            'sources': sources,
            'query': question
        }
        
    except Exception as e:
        logger.error(f"Error answering question: {str(e)}")
        raise

In [20]:
def evaluate_retrieval(self, test_queries: List[str], 
                          ground_truth: List[List[str]] = None) -> Dict[str, float]:
    """Evaluate retrieval performance"""
    try:
        if not self.retriever:
            raise ValueError("Retriever not created")
        
        # Simple evaluation - count relevant documents retrieved
        results = []
        for query in test_queries:
            docs = self.retriever.get_relevant_documents(query)
            results.append(len(docs))
        
        avg_docs_retrieved = np.mean(results)
        
        return {
            'avg_documents_retrieved': avg_docs_retrieved,
            'total_queries': len(test_queries)
        }
        
    except Exception as e:
        logger.error(f"Error evaluating retrieval: {str(e)}")
        raise